# `system_one` over vLLM

The same questions as `system_one.ipynb`, asked through `jev_vllm.client` of an exported `pointer` run that vLLM
serves (`uv run serve`). The client is shaped like `typesafe-sdk`'s: the same `system_one(state, questions)`, the SDK's
question types in, and its `SystemOneResponse` and `TypeSafe*Error`s out.

- `Noul(instructions=...)`: a yes/no question; the answer is the probability of yes.
- `Choice(instructions=..., criteria=...)`: one label out of several. `criteria` maps each label to when it applies, or
  to `None` when the label speaks for itself; the answer is the top label, its confidence and every label's
  probability.

All questions of one call go to the server in one request and are answered in one batch. The client imports neither
vLLM nor torch, so it runs anywhere `typesafe-sdk` and `httpx` are installed.

## Start the server

In a terminal, serve a `jev-model export` folder or a Hub repo pushed from one (a private repo needs `HF_TOKEN`):

```bash
uv run serve pointer-no-bench-bf16
# uv run serve bizhmans/jev-pointer-no-bench-bf16
```

It listens on `http://localhost:8080/pooling` under the model name `pointer`.

In [ ]:
from typesafe_sdk import Choice, ChoiceAnswer, Noul, NoulAnswer, SystemOneResponse

from jev_vllm.client import AsyncJevClient, JevClient

BASE_URL = "http://localhost:8080"

client = JevClient(base_url=BASE_URL)  # model="pointer", timeout=60.0

## Yes/no questions

A `Noul` has only instructions. Its options are always `yes` and `no`, whatever the language of the instructions.
`response.nouls` holds the yes/no answers by name.

In [ ]:
state = (
    "Hi, I ordered a blender two weeks ago (order #4821) and it still hasn't arrived. "
    "The tracking page has said 'in transit' for nine days. I'd like a refund if it can't be delivered by Friday."
)

response = client.system_one(
    state=state,
    questions={
        "complaint": Noul(instructions="Is the customer complaining?"),
        "wants_refund": Noul(instructions="Does the customer ask for a refund, at least conditionally?"),
        "gave_order_id": Noul(instructions="Does the message include an order number?"),
        "is_spam": Noul(instructions="Is this message spam?"),
    },
)
for name, a in response.nouls.items():
    print(f"{name:15} P(yes) = {a.noul:.3f}")

## Choice questions

Each criterion becomes an option `label: description`, or just `label` when the description is `None`.
A choice needs at least two criteria. `response.choices` holds the choice answers by name.

In [ ]:
response = client.system_one(
    state=state,
    questions={
        "topic": Choice(
            instructions="What is the message mainly about?",
            criteria={
                "delivery": "shipping, tracking or a late or lost parcel",
                "billing": "charges, invoices or payment methods",
                "product": "how a product works or a defect in it",
                "account": "logging in, passwords or profile settings",
            },
        ),
        "sentiment": Choice(
            instructions="What is the customer's tone?", criteria={"positive": None, "neutral": None, "negative": None}
        ),
    },
)
for name, a in response.choices.items():
    probs = ", ".join(f"{label} {p:.2f}" for label, p in a.probabilities.items())
    print(f"{name:10} -> {a.choice} ({a.confidence:.2f})   [{probs}]")

## Mixing both kinds

One call can hold both kinds. `response.answers` has every answer by name; check the type to read it.

In [ ]:
state = "Alice is taller than Bob. Bob is taller than Carol. Dan is shorter than Carol. Erin is taller than Alice."
people = dict.fromkeys(["Alice", "Bob", "Carol", "Dan", "Erin"])

response = client.system_one(
    state=state,
    questions={
        "alice_over_dan": Noul(instructions="Is Alice taller than Dan?"),
        "carol_over_erin": Noul(instructions="Is Carol taller than Erin?"),
        "tallest": Choice(instructions="Who is the tallest?", criteria=people),
        "shortest": Choice(instructions="Who is the shortest?", criteria=people),
    },
)


def show(response: SystemOneResponse) -> None:
    for name, a in response.answers.items():
        if isinstance(a, NoulAnswer):
            print(f"{name:16} yes {a.noul:.3f}")
        elif isinstance(a, ChoiceAnswer):
            print(f"{name:16} {a.choice} ({a.confidence:.2f})")


show(response)

## Plain dicts and JSON states

Questions can also be dicts in the wire shape, and a state that is not a string (a dict or a list) is sent as compact
JSON.

In [ ]:
response = client.system_one(
    state={"order": 4821, "status": "in transit", "days_in_transit": 9, "promised_days": 5},
    questions={
        "late": {"type": "noul", "instructions": "Is the order later than promised?"},
        "status": {
            "type": "choice",
            "instructions": "Where is the order?",
            "criteria": {"warehouse": None, "in transit": None, "delivered": None},
        },
    },
)
show(response)

## A row from a built dataset

States from `jev prepare` work as they are. Here the dataset's options become `Choice` labels with no
description, so the model reads them exactly as in training and the answer can be checked against the label.

In [ ]:
from datasets import load_from_disk

from jev_model.constants import DATA_DIR

NAME = "qasc"  # any dataset built under DATA_DIR
row = load_from_disk(str(DATA_DIR / NAME))["test"][0]

texts = [option["text"] for option in row["options"]]
# labels are dict keys, so repeated option texts would collapse
assert len(set(texts)) == len(texts)
response = client.system_one(
    state=row["state"], questions={"q": Choice(instructions=row["question"], criteria=dict.fromkeys(texts))}
)
a = response.choices["q"]

gold = texts[max(range(len(texts)), key=row["label"].__getitem__)]
print(row["state"], "")
print(row["question"])
print("predicted:", a.choice, f"({a.confidence:.2f})")
print("gold:     ", gold)

## Many calls at once

`AsyncJevClient` has the same `system_one`, awaited. vLLM batches concurrent requests itself, so `asyncio.gather`
over many calls keeps the GPU busy. Jupyter already runs an event loop, so the cell awaits directly.

In [ ]:
import asyncio

messages = [
    "Where is my parcel? It was due on Monday.",
    "You charged my card twice for the same order.",
    "The blender makes a grinding noise when it starts.",
    "I can't log in since I changed my password.",
]
topics = {"delivery": None, "billing": None, "product": None, "account": None}

async with AsyncJevClient(base_url=BASE_URL) as aclient:
    responses = await asyncio.gather(
        *(
            aclient.system_one(
                state=message,
                questions={"topic": Choice(instructions="What is the message about?", criteria=topics)},
            )
            for message in messages
        )
    )
for message, response in zip(messages, responses):
    print(f"{response.choices['topic'].choice:9} {message}")

## Errors

The client checks questions before sending anything and raises `ValueError` or `TypeError`. What the server rejects
comes back as the SDK's errors: `TypeSafeBadRequestError` (400) for a request the plugin can't answer,
`TypeSafeNotFoundError` (404) for an unknown model name, and `TypeSafeAPIConnectionError` when no server is listening.

In [ ]:
from typesafe_sdk import TypeSafeAPIConnectionError, TypeSafeAPIError

for questions in [
    {},
    {"one": Choice(instructions="Pick one", criteria={"only": None})},
    {"bad": "Is it raining?"},
]:
    try:
        client.system_one(state="It is raining.", questions=questions)
    except (ValueError, TypeError) as e:
        print(f"{type(e).__name__}:", e)

try:
    client.system_one(state="It is raining.", questions={"rain": Noul(instructions="Is it raining?")}, model="nope")
except TypeSafeAPIError as e:
    print(f"{type(e).__name__}:", e)

with JevClient(base_url="http://localhost:1") as offline:
    try:
        offline.system_one(state="It is raining.", questions={"rain": Noul(instructions="Is it raining?")})
    except TypeSafeAPIConnectionError as e:
        print(f"{type(e).__name__}:", e)

In [ ]:
client.close()